 # Hypothesis Testing for Income Census data

In [2]:
# import required libraries

import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

In [3]:
# dataset path
file_path = "../income_dataset/"

In [4]:
# declare the column names test and train files dont have headers.
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

# load the data
df_train = pd.read_csv(file_path + "adult.data", header=None, names=columns, sep=",",skipinitialspace=True)
df_test = pd.read_csv(file_path + "adult.test", header=None, names=columns, sep=",", skipinitialspace=True, skiprows=1)

# concatenate both datasets
df = pd.concat([df_train,df_test],axis = 0 , ignore_index=True)

In [5]:
print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
df.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [7]:
print(df['income'].value_counts())

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64


In [8]:
# standardize the income caegories

df['income'] = (df['income'].str.strip().str.replace('.','',regex=False))

In [9]:
print(df['income'].nunique())
print(df['income'].value_counts())

2
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


# hypothesis question : Age vs Income

In [10]:
# hypothesis question
# Does age affect the probability of Income
# H0 - Age does not affect the probability of earning > 50K
# H1 - Age significantly affects the probability of earngin > 50K



# 1 = >50K, 0 = <=50K
df['income_binary'] = (df['income'] == '>50K').astype(int)


model = smf.logit (formula = "income_binary ~ age",data=df)
result = model.fit()
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.524280
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:          income_binary   No. Observations:                48842
Model:                          Logit   Df Residuals:                    48840
Method:                           MLE   Df Model:                            1
Date:                Mon, 09 Feb 2026   Pseudo R-squ.:                 0.04720
Time:                        11:59:20   Log-Likelihood:                -25607.
converged:                       True   LL-Null:                       -26875.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7228      0.035    -78.308      0.000      -2.791      -2.655
age            0.0387      0.

#### - Convert logistic regression coefficient from log-odds to odds ratio
#### - In logistic regression, coef represents change in log-odds of the outcome per unit increase in predictor
#### - np.exp(coef) gives the odds ratio, which is easier to interpret:
####   - OR > 1 means higher odds of outcome as predictor increases
####   - OR < 1 means lower odds
#### - Example: coef = 0.0387 → odds ratio ≈ 1.0395 --> each 1-year increase in age raises odds of earning >50K by ~3.95%

In [11]:
np.exp(0.0387)

1.039458599290056

#### Results : 
Age coefficient = 0.0387 - Each year of age increases the probability of earning > 50K by about 3.95%
Pseudo R² = 0.0472 - Age alone explains ~4.7% of the variation in income
p-value for age = 0.000  < 0.05 --> reject H0 null hypothesis. age is statistically significant
Conclusion : age is significant preditor of income, older individuals having higher probability of earning > 50K

---
---

# Hypothesis 2: Race vs Income
#### H0: Race has no effect on income (person’s race does not change the likelihood of earning >50K or ≤50K).
#### H1 : Race has an effect on income (person’s race changes the likelihood of earning >50K or ≤50K).


In [12]:

# Create a contingency table
contingency = pd.crosstab(df['race'], df['income'])

# Perform Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)

# Print results
print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:\n", expected)

# Interpretation
if p < 0.05:
    print("Reject H0: Race and income are not independent (significant relationship).")
else:
    print("Fail to reject H0: No significant relationship between race and income.")


Chi-square statistic: 487.026286837627
p-value: 4.284377710223499e-104
Degrees of freedom: 4
Expected frequencies:
 [[  357.53757012   112.46242988]
 [ 1155.53099791   363.46900209]
 [ 3563.96492773  1121.03507227]
 [  308.85160313    97.14839687]
 [31769.11490111  9992.88509889]]
Reject H0: Race and income are not independent (significant relationship).


#### -Race vs Income Analysis
#### -The Chi-square test shows a very strong association between race and income (Chi-square = 487.03, df = 4, p < 0.001).
#### -The expected frequencies (the counts we would expect if race had no effect on income) differ from the actual observed counts, which is why the test is highly significant.
#### -Conclusion: Race has a significant effect on income — the likelihood of earning >50K varies across different racial groups.

#### - Degrees of freedom tell us how many independent pieces of information are used to calculate the Chi-square statistic.
#### - dof=(number of rows−1)×(number of columns−1)
#### -      Rows = number of categories in one variable (e.g., races)
#### -      Columns = number of categories in the other variable (e.g., income groups)

---

# Hypothesis : Marital status vs Income
#### - H0 : Income level is independent of marital status
#### - H1 : Income level is dependednt of marital status

In [13]:
# Create contingency table
income_cont = pd.crosstab(df['marital_status'], df['income'])

#chi-square test
chi2, p, dof, expected = chi2_contingency(income_cont)

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

Chi-square statistic: 9816.015037266438
p-value: 0.0
Degrees of freedom: 6


#### - Since the p-value is less than 0.05, we reject the null hypothesis. 
#### - There is a statistically significant association between marital status and income level.
#### - Income (>50K vs ≤50K) differs significantly across marital-status categories.

# Hypothesis : Country and education levels association with Income


In [14]:
# First, define the groups
low_edu = [
    'Preschool','1st-4th','5th-6th','7th-8th',
    '9th','10th','11th','12th','HS-grad'
]

high_edu = [
    'Bachelors','Prof-school','Assoc-acdm',
    'Assoc-voc','Masters','Doctorate'
]

# Create a new column in the dataframe
df['education_group'] = df['education'].apply(
    lambda x: 'low_edu' if x in low_edu else ('high_edu' if x in high_edu else 'Other')
)
print(df['education_group'].value_counts())



education_group
low_edu     22192
high_edu    15772
Other       10878
Name: count, dtype: int64


In [15]:


model = smf.logit(
    "income_binary ~ C(education_group) + C(native_country)",
    data=df
).fit()

print(model.summary())
'''odds_ratios = pd.DataFrame({
    "OR": np.exp(model.params),
    "Lower 95% CI": np.exp(model.conf_int()[0]),
    "Upper 95% CI": np.exp(model.conf_int()[1]),
    "p-value": model.pvalues
})
print(odds_ratios)'''

Optimization terminated successfully.
         Current function value: 0.500619
         Iterations 33
                           Logit Regression Results                           
Dep. Variable:          income_binary   No. Observations:                48842
Model:                          Logit   Df Residuals:                    48798
Method:                           MLE   Df Model:                           43
Date:                Mon, 09 Feb 2026   Pseudo R-squ.:                 0.09020
Time:                        11:59:32   Log-Likelihood:                -24451.
converged:                       True   LL-Null:                       -26875.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept                        

'odds_ratios = pd.DataFrame({\n    "OR": np.exp(model.params),\n    "Lower 95% CI": np.exp(model.conf_int()[0]),\n    "Upper 95% CI": np.exp(model.conf_int()[1]),\n    "p-value": model.pvalues\n})\nprint(odds_ratios)'

#### Higher education significantly increases the likelihood of earning more than 50K, regardless of country.
#### Country also affects income probabilities, with some countries showing higher or lower odds.
#### Overall, education is the strongest predictor of income in this dataset